# DADA-2000 original — Phase 5 / **Option A: z-scored flow target (`flow/v2_zscore`)**

Plan: `.project/plans/katvad-flow-zscore-option-a.md` · Parent: `katvad-kip-loss-scale-diagnosis.md` §5 **A**
(licensed by D1/D2, authorized 2026-09-23, construction **A1**).

**What changes, and nothing else:** KIP-on arms train on `e_O = ((s − μ)/σ) @ M` — the same 23 raw flow
stats, standardized with **T2-train-window** moments, through the **same** projection `M` — with
`loss.lambda_rec = 1/V` **derived** on the new cache (never swept, lesson 14). Every other flag is phase 4's.

| phase | where | exit |
|---|---|---|
| **P0** §1 | CPU | v1 flow cache complete for every train source; projection present |
| **P2** §2 | CPU, minutes | `core.flow.zscore_cache` passes G0–G2; `lambda_rec` read from its manifest |
| **P3** §3–§4 | GPU | KIP-on stage 1 + stage 2 × seeds 2024/2025/2026 on v2 |
| **P4** §5–§8 | GPU | eval T2 + DoTA; R-1, R-1b, R-2, R-3 scored against the table **written before this ran** |

> **C2 fires.** Every KIP-on number measured on `flow/v1` is superseded *for this question*; it is not
> overwritten — v1 stays on disk and stays the record of phase 4. **KIP-off arms are reused from phase 4**,
> not re-run: KIP-off never reads flow or `lambda_rec` (pinned by
> `core/tests/test_flow_zscore.py::TestKipOffIsInert`). §4 checks the configs differ in exactly the
> expected keys before the pairing is used.

> **C24 is untouched.** On `main` the gate MLP receives no gradient; every KIP-on arm is a fixed ~50 %
> channel shift. A positive result here is attributable to the **loss target**, never to motion gating.

**Upload the P1 code first** (`core/flow/zscore.py`, `core/flow/zscore_cache.py`, the edited
`core/eda/features.py` and `core/constants.py`) — §0 refuses to start without it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'

os.environ['REPO'] = f'{DRIVE}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'flow' / 'zscore_cache.py').is_file(), (
    f'{REPO} predates Option A -- upload core/flow/zscore.py, core/flow/zscore_cache.py and the '
    'edited core/eda/features.py + core/constants.py (plan P1) before running this notebook')
os.environ['PYTHONPATH'] = str(REPO)
sys.path.insert(0, str(REPO))

from core import constants  # noqa: E402

DATASET = constants.DADA_ORIGIN_DATASET
T2 = constants.DATA_ROOT / DATASET
CLIP_DIR = constants.CLIP_CACHE_DIR / DATASET
FLOW_V1_ROOT = constants.FLOW_CACHE_DIR                    # read only -- never written here
FLOW_V2_ROOT = constants.FLOW_ZSCORE_CACHE_DIR             # written by section 2
KNN_CACHE = constants.KNN_CACHE_DIR / DATASET / constants.KNN_CACHE_FILENAME
RUNS4 = constants.OUTPUT_ROOT / f'{DATASET}_phase4'        # phase 4: KIP-off arms reused from here
RUNS = constants.OUTPUT_ROOT / f'{DATASET}_zscore'         # every Option-A arm lands here
P5 = Path('/content/p5z')                                  # VM-local scratch, never Drive
P5.mkdir(parents=True, exist_ok=True)

DOTA_DATA = constants.DATA_ROOT / 'DoTA' / 'labels_s8'
DOTA_CLIP = constants.CLIP_CACHE_DIR / 'DoTA_s8_ncc'

# --- identical to phase_4.ipynb, flag for flag: the pairing depends on it -----------
SEEDS = (2024, 2025, 2026)
KERNEL = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL   # 3  (C27)
TOPK_PCT = constants.DADA_ORIGIN_MIL_TOPK_PCT      # 5  (declared deviation, phase-2 plan 6.2.1)
NUM_EPOCHS = 20                                    # 2,040 steps at 102 steps/epoch
CHUNK_EPOCHS = 5

# --- the pre-registered read-out bars (plan 6.3), as code ---------------------------
R2_RHO_CAPTURE = 1.0          # R-2: rho < 1.0 on every seed mean -> capture removed
T95 = {2: 12.706, 3: 4.303}   # two-sided t, n-1 df
V1 = {'V': 31.642, 'K': 11.620, 'rho_end': 3.105, 'lambda_rec': 1.0}   # D1/D2, for the record only

print(f'dataset {DATASET} | kernel {KERNEL} | mil_topk_pct {TOPK_PCT} | {NUM_EPOCHS} epochs')
for name, path in (('corpus', T2), ('clip', CLIP_DIR), ('flow v1', FLOW_V1_ROOT / DATASET),
                   ('flow v2', FLOW_V2_ROOT / DATASET), ('knn', KNN_CACHE),
                   ('phase-4 runs', RUNS4), ('runs', RUNS),
                   ('DoTA data', DOTA_DATA), ('DoTA clip', DOTA_CLIP)):
    print(f'  {name:13s} {path}   {"OK" if path.exists() else "MISSING"}')

In [ ]:
%%bash
# Same pins as phase_4.ipynb (C7: transformers internals on the text path).
pip install -q "transformers==4.56.*" av einops faiss-cpu
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- sections 3-7 will be unusably slow'
df -h /content | tail -1

### 0.1 Stage to VM-local disk

`core/train.py` reads one `.npy` per item per step from the dataset, serially; on a Drive FUSE mount each
is a network round trip. Copies are bit-identical — no transform changes (C2).

In [ ]:
import json
import shutil
import subprocess
import time

STAGE = P5 / 'stage'
STAGE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, 'PYTHONPATH': str(REPO)}


def stage(src, name):
    # Copy a Drive path to VM-local disk once; return the local path.
    if src is None or not src.exists():
        return src
    dst = STAGE / name
    if dst.exists():
        print(f'  {name:14s} already staged')
        return dst
    t0 = time.time()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    files = [f for f in dst.rglob('*') if f.is_file()] if dst.is_dir() else [dst]
    mib = sum(f.stat().st_size for f in files) / 2**20
    print(f'  {name:14s} {len(files):5d} files, {mib:7.1f} MiB in {time.time() - t0:5.1f}s')
    return dst


def run(cmd, cwd=REPO, capture=False):
    # Run a child; make its failure legible instead of a bare CalledProcessError.
    proc = subprocess.run([str(c) for c in cmd], cwd=str(cwd), env=ENV,
                          capture_output=capture, text=True)
    if proc.returncode:
        if capture:
            print(proc.stdout or '', proc.stderr or '', sep='\n')
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd)}'
                           + ('' if capture else '  -- scroll up for the child output'))
    if capture and proc.stdout:
        print(proc.stdout.rstrip())


print('staging to VM-local NVMe (bit-identical copies; C2 untouched):')
L_CLIP = stage(CLIP_DIR, 'clip')
L_T2 = stage(T2, 'corpus')
L_KNN = stage(KNN_CACHE, 'knn_cache.npz')
L_DOTA_CLIP = stage(DOTA_CLIP, 'dota_clip')
L_DOTA_DATA = stage(DOTA_DATA, 'dota_data')

# The v1 ROOT, not just the dataset dir: zscore_cache reads flow_projection.npz beside it.
L_V1_ROOT = STAGE / 'flow_v1'
stage(FLOW_V1_ROOT / DATASET, f'flow_v1/{DATASET}')
stage(FLOW_V1_ROOT / constants.FLOW_PROJECTION_FILENAME,
      f'flow_v1/{constants.FLOW_PROJECTION_FILENAME}')

## 1. **P0** — the v1 cache is complete for what Option A reads

Containment, not a count (pending.md 2026-09-16): an equal number of files over the wrong ids passes a
count and then fails per item inside training. Every train **source** needs both `{id}.npy` (for G0) and
`{id}.stats.npy` (the input).

In [ ]:
from core.eda.corpus import load_dataset_files

FILES = load_dataset_files(L_T2, DATASET)
train_windows = FILES.train_ids
train_sources = sorted({FILES.source_of(w) for w in train_windows})
lengths = {w.length for w in (FILES.windows or {}).values()}
assert lengths == {constants.DADA_ORIGIN_WINDOW_LENGTH}, (
    f'corpus windows {sorted(lengths)} != W={constants.DADA_ORIGIN_WINDOW_LENGTH}: not the Gate-W build')

v1_dir = L_V1_ROOT / DATASET
suffix = constants.FLOW_STATS_SUFFIX
have_stats = {p.name[:-len(suffix)] for p in v1_dir.glob(f'*{suffix}')}
have_e_o = {p.stem for p in v1_dir.glob('*.npy') if not p.name.endswith(suffix)}
projection_ok = (L_V1_ROOT / constants.FLOW_PROJECTION_FILENAME).is_file()

print(f'train windows {len(train_windows):,} over {len(train_sources):,} sources')
print(f'v1 {v1_dir}:  e_O {len(have_e_o):,} | .stats.npy {len(have_stats):,} | '
      f'projection {"OK" if projection_ok else "MISSING"}')
missing_stats = sorted(set(train_sources) - have_stats)
missing_e_o = sorted(set(train_sources) - have_e_o)
orphans = sorted(have_stats - have_e_o)
assert projection_ok, 'flow_projection.npz is missing -- the v1 cache is unusable (raft_extract docstring)'
assert not missing_stats, f'{len(missing_stats)} train sources lack .stats.npy: {missing_stats[:5]}'
assert not missing_e_o, f'{len(missing_e_o)} train sources lack e_O: {missing_e_o[:5]}'
assert not orphans, f'{len(orphans)} sources have stats but no e_O: {orphans[:5]} (zscore_cache refuses them too)'
print('P0 PASS -- every train source has e_O + raw stats; the projection is present')

## 2. **P2** — build `flow/v2_zscore` and score the build gates

`core.flow.zscore_cache` fits μ, σ on the **train windows** (the rows `L_KIP_rec` averages over, the
population D1's `V` was measured on), G0-checks every v1 file against `stats @ M`, writes the z-scored
`e_O` atomically, copies `.stats.npy` byte-identical, then scores G1/G2 on the new cache and writes
`lambda_rec = round(1/V, 4)` into `zscore_manifest.json`. **It exits non-zero on any HARD gate, after
writing the manifest.**

| id | bar | on fail |
|---|---|---|
| G0 | every v1 `e_O` == its `stats @ M` (rtol 1e-4, atol 1e-3) | HARD STOP |
| G0-c | v2 file count == v1 file count | HARD STOP |
| G1 | standardized train stats: \|mean\| ≤ 1e-3, std ∈ [0.999, 1.001] | HARD STOP |
| G2-a | `V_v2` ∈ [0.80, 1.25] (predicted ≈ 1) | report, never re-weight |
| G2-b | `Z_v2 / V_v2` ∈ [0.99, 1.01] | HARD STOP |
| G2-c | round-trip ∈ [0.9, 1.1] — on v2 this is ≈ `1/V_v2` (plan §6.1 clarification) | HARD STOP |
| G2-d/e | between-item share (v1 0.488), `W/V` | reported |

Built on local disk, then copied to Drive (C10: count files after).

In [ ]:
L_V2_ROOT = STAGE / 'flow_v2_zscore'
L_FLOW = L_V2_ROOT / DATASET
MANIFEST_NAME = constants.FLOW_ZSCORE_MANIFEST_FILENAME

drive_manifest = FLOW_V2_ROOT / DATASET / MANIFEST_NAME
if drive_manifest.is_file() and not (L_FLOW / MANIFEST_NAME).is_file():
    print('v2 already built on Drive -- staging it instead of rebuilding')
    stage(FLOW_V2_ROOT / DATASET, f'flow_v2_zscore/{DATASET}')
    stage(FLOW_V2_ROOT / constants.FLOW_PROJECTION_FILENAME,
          f'flow_v2_zscore/{constants.FLOW_PROJECTION_FILENAME}')

# Resumable: rerunning skips finished files and re-verifies the gates.
run([sys.executable, '-m', 'core.flow.zscore_cache',
     '--data-dir', L_T2, '--dataset', DATASET,
     '--src-root', L_V1_ROOT, '--dst-root', L_V2_ROOT])

ZM = json.loads((L_FLOW / MANIFEST_NAME).read_text(encoding='utf-8'))
print(f"\ncoverage {ZM['coverage']} | G0 max |err| this run {ZM['g0']['max_abs_err']:.3g}")
print(f"fitted on {ZM['zscore']['n_items']:,} train windows / {ZM['zscore']['n_rows']:,} rows")
print(f'{"gate":24s} {"verdict":8s} value')
for name, gate in ZM['gates'].items():
    word = 'PASS' if gate['passed'] else ('FAIL' if gate['hard'] else 'NOTE')
    print(f'{name:24s} {word:8s} {gate["value"]}   (bar {gate["bar"]})')
assert ZM['passed'], 'a HARD gate failed -- stop here and bring the manifest back (plan 6.1)'

T_V2 = ZM['target']
V2 = T_V2['mse_global_mean_predictor']
LAMBDA_REC = ZM['lambda_rec']
print(f"\nV_v2 {V2:.4f} (v1 {V1['V']}) | Z_v2 {T_V2['mse_zero_predictor']:.4f} | "
      f"W_v2 {T_V2['mse_item_mean_predictor']:.4f} | between-item {T_V2['between_item_share']:.3f}")
print(f'lambda_rec = 1/V_v2 = {LAMBDA_REC}   (DERIVED; v1 used {V1["lambda_rec"]} on a V of {V1["V"]})')
assert LAMBDA_REC is not None and abs(LAMBDA_REC - 0.0316) > 0.01, (
    'lambda_rec came out at v1\'s 1/V -- that is the wrong cache\'s weight and would switch L_KIP_rec off')

### 2.1 Independent cross-check with the EDA report, then persist v2 to Drive

The same numbers through `core.tools.eda report --sections features` — the tool that produced D1. On v2
it sees `zscore_stats.npz` and predicts the round-trip from the **standardized** stats (plan D-6);
`target_normalized` must be `true`.

In [ ]:
EDA_OUT = RUNS / 'p2_flow_target_v2'
run([sys.executable, '-m', 'core.tools.eda', 'report',
     '--dataset', DATASET, '--data-dir', L_T2, '--clip-dir', L_CLIP,
     '--flow-dir', L_FLOW, '--output-dir', EDA_OUT,
     '--sections', 'features', '--no-probe'], capture=True)
eda_flow = json.loads((EDA_OUT / constants.EDA_REPORT_JSON_FILENAME).read_text())['features']['flow']
assert eda_flow['target_normalized'] is True, 'eda did not see zscore_stats.npz -- stale checkout?'
eda_V = eda_flow['target']['mse_global_mean_predictor']
print(f'eda V_v2 {eda_V:.6f} vs tool {V2:.6f}')
assert abs(eda_V - V2) <= 1e-6 * max(1.0, V2), 'the two readings of V_v2 disagree'

# Persist: local -> Drive, then count (C10). v1 is never touched.
dst = FLOW_V2_ROOT / DATASET
dst.mkdir(parents=True, exist_ok=True)
shutil.copy2(L_V2_ROOT / constants.FLOW_PROJECTION_FILENAME,
             FLOW_V2_ROOT / constants.FLOW_PROJECTION_FILENAME)
run(['rsync', '-a', f'{L_FLOW}/', f'{dst}/'])
n_local = sum(1 for _ in L_FLOW.iterdir())
n_drive = sum(1 for _ in dst.iterdir())
print(f'v2 on Drive: {n_drive} files (local {n_local}) at {dst}')
assert n_drive == n_local, 'Drive copy is short -- rerun this cell'

## 3. **P3** — train the KIP-on arms on v2

Phase 4's `train_cmd`, byte for byte, plus exactly two differences: `--flow-dir` → v2, and
`--set loss.lambda_rec=<derived>`. Stage 1 (KIP warm-up) then stage 2 (warm-started via
`--init-weights`), 3 seeds, 5-epoch chunks synced to Drive so a disconnect costs ≤ 5 epochs.

In [ ]:
def train_cmd(out_dir, *, seed, stage, init_weights=None):
    # phase_4.ipynb train_cmd(kip_on=True) + the two Option-A differences.
    cmd = [sys.executable, '-m', 'core.train',
           '--set', f'train.stage={stage}',
           '--set', f'train.seed={seed}',
           '--set', f'train.num_epochs={NUM_EPOCHS}',
           '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           '--set', f'model.score_head_kernel={KERNEL}',
           '--set', f'loss.mil_topk_pct={TOPK_PCT}',
           # --- Option A: the ONLY two differences from phase 4's KIP-on arms ---
           '--set', f'loss.lambda_rec={LAMBDA_REC}',
           '--flow-dir', L_FLOW,
           '--data-dir', L_T2, '--clip-dir', L_CLIP,
           '--knn-cache', L_KNN,
           '--output-dir', out_dir]
    if init_weights is not None:
        cmd += ['--init-weights', init_weights]
    return cmd


def stage1_dir(seed):
    return RUNS / f's{seed}' / 'stage1'


def arm_dir(seed):
    return RUNS / f's{seed}' / 'stage2_kip_on'


def local_dir(drive_dir):
    return P5 / 'runs' / drive_dir.relative_to(RUNS)


def sync_to_drive(local, drive):
    drive.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        if (local / name).exists():
            shutil.copy2(local / name, drive / name)


def restore_from_drive(drive, local):
    if not drive.exists():
        return
    local.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        if (drive / name).exists() and not (local / name).exists():
            shutil.copy2(drive / name, local / name)


def ckpt_of(run_dir):
    # The VM-local checkpoint if this session trained it, else the Drive copy (after a restart).
    local = local_dir(run_dir) / 'checkpoint_last.pt'
    return local if local.exists() else run_dir / 'checkpoint_last.pt'


def epochs_done(run_dir):
    # From metrics.jsonl, not torch.load (C15).
    path = run_dir / 'metrics.jsonl'
    if not path.exists():
        return 0
    last = -1
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            last = max(last, json.loads(line)['epoch'])
    return last + 1


def train_arm(drive_out, *, seed, stage, init_weights=None):
    local = local_dir(drive_out)
    restore_from_drive(drive_out, local)
    while True:
        done = epochs_done(local)
        if done >= NUM_EPOCHS:
            print(f'  {drive_out.parent.name}/{drive_out.name}: {done}/{NUM_EPOCHS} epochs -- done')
            sync_to_drive(local, drive_out)
            return
        cmd = train_cmd(local, seed=seed, stage=stage, init_weights=None if done else init_weights)
        if done:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, NUM_EPOCHS - done)]
        print(f'  {drive_out.parent.name}/{drive_out.name}: epochs {done} -> '
              f'{min(done + CHUNK_EPOCHS, NUM_EPOCHS)} of {NUM_EPOCHS}')
        run(cmd)
        sync_to_drive(local, drive_out)


for seed in SEEDS:
    print(f'=== seed {seed}: stage 1 (KIP warm-up, v2 target)')
    train_arm(stage1_dir(seed), seed=seed, stage=1)

In [ ]:
for seed in SEEDS:
    init = ckpt_of(stage1_dir(seed))
    assert init.exists(), f'seed {seed}: stage 1 checkpoint missing at {init}'
    print(f'=== seed {seed}: stage 2, KIP on, v2 target')
    train_arm(arm_dir(seed), seed=seed, stage=2, init_weights=init)

## 4. The pairing is what we say it is

The paired Δ reuses phase 4's **KIP-off** arms. That is sound only if this run's stage-2 config differs
from phase 4's **KIP-on** config in `loss.lambda_rec` and nothing else (the flow dir is a CLI path, not a
config key), and phase 4's KIP-off differs from phase 4's KIP-on in `kip.enabled` only. Anything else
found here means the Δ is not the one pre-registered — stop and re-run KIP-off on this config.

In [ ]:
import yaml


def flat(tree, prefix=''):
    out = {}
    for key, value in tree.items():
        name = f'{prefix}{key}'
        if isinstance(value, dict):
            out.update(flat(value, f'{name}.'))
        else:
            out[name] = value
    return out


def diff(a, b):
    fa, fb = flat(yaml.safe_load(a.read_text())), flat(yaml.safe_load(b.read_text()))
    return {k: (fa.get(k), fb.get(k)) for k in sorted(set(fa) | set(fb)) if fa.get(k) != fb.get(k)}


for seed in SEEDS:
    new = arm_dir(seed) / 'config.yaml'
    old_on = RUNS4 / f's{seed}' / 'stage2_kip_on' / 'config.yaml'
    old_off = RUNS4 / f's{seed}' / 'stage2_kip_off' / 'config.yaml'
    d_new = diff(old_on, new)
    d_off = diff(old_on, old_off)
    print(f's{seed}  v1-on -> v2-on : {d_new}')
    print(f's{seed}  v1-on -> off   : {d_off}')
    assert set(d_new) == {'loss.lambda_rec'}, f'seed {seed}: v2 run differs in {sorted(d_new)}'
    assert float(d_new['loss.lambda_rec'][1]) == LAMBDA_REC, 'config did not record the derived weight'
    assert set(d_off) == {'kip.enabled'}, f'seed {seed}: phase-4 pair differs in {sorted(d_off)}'
print('\npairing OK: v2-on vs off differs in kip.enabled + lambda_rec, and lambda_rec is inert under '
      'KIP-off (TestKipOffIsInert)')

## 5. **P4** — evaluation

Same protocol as phase 4 §6: T2 in-domain (`--score-norm auto` → `none`) and DoTA zero-shot
(`auto` → per-clip min-max, C8). **No `--set model.*` / `kip.*`** — the checkpoint defines the
architecture (C34). KIP-off results are read from phase 4's eval dirs, not recomputed.

In [ ]:
def evaluate(ckpt, out_dir, *, dataset, data_dir, clip_dir):
    if (out_dir / 'results.json').exists():
        return json.loads((out_dir / 'results.json').read_text())
    run([sys.executable, '-m', 'core.evaluate',
         '--ckpt', ckpt,
         '--set', f'data.dataset={dataset}',
         '--data-dir', data_dir, '--clip-dir', clip_dir,
         '--output-dir', out_dir, '--save-scores'])
    return json.loads((out_dir / 'results.json').read_text())


def read_results(path):
    return json.loads(path.read_text()) if path.exists() else None


EVALS = {}
for seed in SEEDS:
    ckpt = ckpt_of(arm_dir(seed))
    EVALS[(seed, 'v2', 't2')] = evaluate(ckpt, RUNS / f's{seed}' / 'eval_t2_kip_on',
                                         dataset=DATASET, data_dir=L_T2, clip_dir=L_CLIP)
    EVALS[(seed, 'v2', 'dota')] = evaluate(ckpt, RUNS / f's{seed}' / 'eval_dota_kip_on',
                                           dataset='DoTA', data_dir=L_DOTA_DATA, clip_dir=L_DOTA_CLIP)
    for arm in ('off', 'on'):                     # phase 4: KIP-off (the pair) and v1 KIP-on (reference)
        for bench in ('t2', 'dota'):
            EVALS[(seed, f'p4_{arm}', bench)] = read_results(
                RUNS4 / f's{seed}' / f'eval_{bench}_kip_{arm}' / 'results.json')
missing = [k for k, v in EVALS.items() if v is None]
assert not any(k[1] == 'p4_off' for k in missing), f'phase-4 KIP-off results missing: {missing}'
print(f'{sum(v is not None for v in EVALS.values())} result sets loaded; missing (reference only): {missing}')

## 6. **R-1 / R-1b** — does PMG fit the standardized target, and which part of it?

* **R-1** `R²_v2 = 1 − kip_rec / V_v2` at the stage-2 end (final-epoch mean of `metrics.jsonl`). Must be
  read against **this** cache's `V`; v1's R² was 0.633 against 31.64.
* **R-1b** (plan risk 1): `M` is `(23, 256)` with rank 23 and `e_O` lies in its row space, so
  `ŝ = ê_O @ pinv(M)` recovers the 23 standardized stats exactly. R² per block — magnitude/u/v moments
  (dims 0–6) vs the 16-bin angle histogram (7–22, **69.6 %** of the target by construction). Scored over
  every train window, model in `eval()`. **Diagnostic only — block re-weighting is not authorized.**

In [ ]:
import statistics

import numpy as np
import torch

from core.config import load_config
from core.flow.raft_extract import load_projection
from core.flow.zscore import load_zscore_stats, standardize
from core.models.kat_vad import KATVAD
from core.train import TEXT_TOWER_PREFIX


def last_epoch_mean(run_dir, key):
    path = run_dir / 'metrics.jsonl'
    if not path.exists():
        return None
    rows = [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]
    final = max(r['epoch'] for r in rows)
    values = [r[key] for r in rows if r['epoch'] == final and key in r]
    return statistics.fmean(values) if values else None


K_V2 = {s: last_epoch_mean(local_dir(arm_dir(s)), 'kip_rec') or last_epoch_mean(arm_dir(s), 'kip_rec')
        for s in SEEDS}
R1 = {s: 1.0 - k / V2 for s, k in K_V2.items()}
for s in SEEDS:
    print(f'R-1  s{s}: kip_rec {K_V2[s]:.4f} / V_v2 {V2:.4f} -> R2 {R1[s]:+.4f}')
print(f'R-1  mean R2 {statistics.fmean(R1.values()):+.4f}   (> 0 expected; v1 was 0.633 on its own V)')

M = load_projection(L_V2_ROOT / constants.FLOW_PROJECTION_FILENAME)          # (23, 256)
PINV = np.linalg.pinv(M.astype(np.float64))                                  # (256, 23)
ZS = load_zscore_stats(L_FLOW / constants.FLOW_ZSCORE_STATS_FILENAME)
slicer = FILES.slicer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BLOCKS = {'magnitude 0-6': slice(0, 7), 'histogram 7-22': slice(7, 23)}
BATCH = 256


def block_r2(seed):
    cfg = load_config(arm_dir(seed) / 'config.yaml', [])
    model = KATVAD.from_config(cfg, load_clip=False)
    payload = torch.load(ckpt_of(arm_dir(seed)),
                         map_location='cpu', weights_only=False)  # own checkpoint
    state = {k: v for k, v in payload['model'].items() if not k.startswith(TEXT_TOWER_PREFIX)}
    model.load_state_dict(state, strict=True)                   # fail loud on anything else (C5)
    model.to(device).eval()
    sse = np.zeros(constants.FLOW_STATS_DIM)
    sst = np.zeros(constants.FLOW_STATS_DIM)
    ids = FILES.train_ids
    for i in range(0, len(ids), BATCH):
        chunk = ids[i:i + BATCH]
        feats = torch.stack([torch.from_numpy(slicer.load(L_CLIP, w).astype(np.float32)) for w in chunk])
        target = np.stack([standardize(slicer.load(L_FLOW, w, constants.FLOW_STATS_SUFFIX), ZS)
                           for w in chunk])                     # (B, L, 23), mean 0 over train
        lengths = torch.full((len(chunk),), feats.shape[1], dtype=torch.long)
        with torch.no_grad():
            eo_hat = model(feats.to(device), lengths.to(device))['eo_hat'].float().cpu().numpy()
        s_hat = eo_hat.astype(np.float64) @ PINV                # (B, L, 23)
        sse += ((s_hat - target) ** 2).reshape(-1, constants.FLOW_STATS_DIM).sum(0)
        sst += (target ** 2).reshape(-1, constants.FLOW_STATS_DIM).sum(0)   # around the train mean (0)
    return {name: float(1.0 - sse[b].sum() / sst[b].sum()) for name, b in BLOCKS.items()} | {
        'all 23': float(1.0 - sse.sum() / sst.sum())}


R1B = {s: block_r2(s) for s in SEEDS}
for s, r in R1B.items():
    print(f'R-1b s{s}: ' + '  '.join(f'{k} {v:+.4f}' for k, v in r.items()))
mean_mag = statistics.fmean(r['magnitude 0-6'] for r in R1B.values())
mean_hist = statistics.fmean(r['histogram 7-22'] for r in R1B.values())
print(f'R-1b mean: magnitude {mean_mag:+.4f} | histogram {mean_hist:+.4f}  -> '
      + ('histogram << magnitude: risk 1 is LIVE (the target is mostly direction noise PMG cannot fit)'
         if mean_hist < 0.5 * mean_mag else 'no block collapse'))

## 7. **R-2** — is the trunk still captured?

`core.tools.grad_probe` exactly as D2 ran it (8 batches of 64, epoch-0 permutation, `autograd.grad` per
weighted term at the temporal encoder), at `stage1` and `stage2_kip_on`, **under this run's config and the
v2 flow dir**. Bar: `rho < 1.0` on every seed mean at the stage-2 end → capture removed. Prediction ≈ 0.5
(target scale ÷ √31.6, weight ≈ ×1). v1: 3.105.

In [ ]:
NUM_BATCHES = 8
D2 = {}
for seed in SEEDS:
    for point, run_dir in (('stage1', stage1_dir(seed)), ('stage2_kip_on', arm_dir(seed))):
        out = RUNS / 'r2_grad' / f's{seed}_{point}'
        path = out / 'grad_probe.json'
        if not path.exists():
            ckpt = ckpt_of(run_dir)
            print(f's{seed} {point}: probing {NUM_BATCHES} batches ...')
            run([sys.executable, '-m', 'core.tools.grad_probe',
                 '--config', arm_dir(seed) / 'config.yaml',
                 '--data-dir', L_T2, '--clip-dir', L_CLIP, '--flow-dir', L_FLOW,
                 '--knn-cache', L_KNN, '--checkpoint', ckpt,
                 '--output-dir', out, '--num-batches', NUM_BATCHES], capture=True)
        D2[(seed, point)] = json.loads(path.read_text(encoding='utf-8'))

RHO = {}
for (seed, point), payload in sorted(D2.items()):
    s = payload['summary']
    rho = s['trunk_kip_total_over_task']
    cos = s['trunk_cosine_with_task'].get('kip_rec', float('nan'))
    vals = [r['trunk']['kip_over_task'].get('kip_rec') for r in payload['records'] if 'trunk' in r]
    vals = [v for v in vals if v is not None]
    sd = statistics.stdev(vals) if len(vals) > 1 else 0.0
    if point == 'stage2_kip_on':
        RHO[seed] = rho
    print(f's{seed} {point:15s} rho(all KIP) {rho:7.3f} | rho(kip_rec) '
          f'{statistics.fmean(vals) if vals else float("nan"):6.3f} +- {sd:5.3f} | cos(kip_rec, task) {cos:+.3f}')

R2_CAPTURED = any(v >= R2_RHO_CAPTURE for v in RHO.values())
print(f'\nR-2  stage-2 rho per seed {dict((s, round(v, 3)) for s, v in RHO.items())} '
      f'(v1 {V1["rho_end"]}) -> ' + ('STILL CAPTURED (>= 1.0 on a seed)' if R2_CAPTURED
                                     else 'capture REMOVED (< 1.0 on every seed)'))

## 8. **R-3 / R-4 / R-5** — the paired Δ and the decision row

Primary **R-3**: paired Δ T2 `auc` micro, KIP-on(v2) − KIP-off(phase 4), t95 at n = 3 — the only row
whose interval excluded zero on v1 (−0.0129 [−0.0249, −0.0008]). **R-4** T2 macro, secondary. **R-5** DoTA:
**reported, never decided on** (v1 intervals were ~20× their own Δ). The v1 KIP-on row is printed beside
it for reference and enters no decision.

In [ ]:
def metric(seed, arm, bench, key):
    res = EVALS.get((seed, arm, bench))
    return float(res[key]) if res is not None and key in res else float('nan')


def paired(arm, bench, key):
    deltas = [metric(s, arm, bench, key) - metric(s, 'p4_off', bench, key) for s in SEEDS]
    deltas = [d for d in deltas if d == d]
    if len(deltas) < 2:
        return None
    mean = statistics.fmean(deltas)
    half = T95[len(deltas)] * statistics.stdev(deltas) / len(deltas) ** 0.5
    return mean, mean - half, mean + half, ''.join('+' if d > 0 else '-' for d in deltas)


print(f'{"row":22s} {"off":>7s} {"v2-on":>7s} {"v1-on":>7s} | {"Δ v2-off":>9s} {"t95":>22s} signs')
ROWS = {}
for rid, bench, key in (('R-3 T2 micro', 't2', 'auc'), ('R-4 T2 macro', 't2', 'auc_macro'),
                        ('R-5 DoTA macro', 'dota', 'auc_macro'), ('R-5 DoTA micro', 'dota', 'auc')):
    off = statistics.fmean(metric(s, 'p4_off', bench, key) for s in SEEDS)
    on2 = statistics.fmean(metric(s, 'v2', bench, key) for s in SEEDS)
    on1 = [metric(s, 'p4_on', bench, key) for s in SEEDS]
    on1 = statistics.fmean(on1) if all(v == v for v in on1) else float('nan')
    p = paired('v2', bench, key)
    ROWS[rid] = {'off': off, 'v2_on': on2, 'v1_on': on1,
                 'delta': p[0] if p else None, 't95': [p[1], p[2]] if p else None,
                 'signs': p[3] if p else None}
    ci = f'[{p[1]:+.4f}, {p[2]:+.4f}]' if p else '-'
    print(f'{rid:22s} {off:7.4f} {on2:7.4f} {on1:7.4f} | {p[0] if p else float("nan"):+9.4f} {ci:>22s} '
          f'{p[3] if p else ""}')

r3 = ROWS['R-3 T2 micro']
lo, hi = r3['t95']
if R2_CAPTURED:
    VERDICT = 'A did not remove capture'
    NEXT = 're-open D2; do NOT lower lambda_rec off a delta'
elif hi < 0:
    VERDICT = 'loss scale was not the (whole) cost'
    NEXT = 'C24 (dead gate -> fixed 50 % shift) is the next suspect -- branch v3; or plan 5 D (conclude)'
elif lo > 0:
    VERDICT = 'A repaired KIP on T2'
    NEXT = 'replicate on a second seed triple before any claim; attribute to the loss target, not motion (C24)'
else:
    VERDICT = 'cost removed; KIP-v1 neutral on T2'
    NEXT = 'write a bounded null; stop spending seeds on v1 KIP'
print(f'\nR-2 {"captured" if R2_CAPTURED else "not captured"} x R-3 t95 [{lo:+.4f}, {hi:+.4f}] '
      f'-> **{VERDICT}**\n  next: {NEXT}')
print('  (v1 R-3 was -0.0129 [-0.0249, -0.0008]; DoTA rows decide nothing at this n)')

## 9. Record the campaign (C17)

`outputs/` is gitignored: this manifest and the copied `results.json` / `config.yaml` /
`zscore_manifest.json` are the only durable record until the numbers reach the plan's Appendix A.

In [ ]:
import hashlib

import transformers

dest = constants.OUTPUT_ROOT / 'REPORTS' / f'{DATASET}_zscore'
dest.mkdir(parents=True, exist_ok=True)
shutil.copy2(L_FLOW / MANIFEST_NAME, dest / MANIFEST_NAME)
for (seed, arm, bench), res in EVALS.items():
    if arm == 'v2' and res is not None:
        (dest / f'results_s{seed}_kip_on_v2_{bench}.json').write_text(json.dumps(res, indent=2))
for seed in SEEDS:
    for name, run_dir in (('stage1', stage1_dir(seed)), ('stage2', arm_dir(seed))):
        if (run_dir / 'config.yaml').exists():
            shutil.copy2(run_dir / 'config.yaml', dest / f'config_s{seed}_{name}.yaml')

digest = hashlib.sha256()
for path in sorted((REPO / 'core').rglob('*.py')):
    digest.update(path.relative_to(REPO).as_posix().encode())
    digest.update(path.read_bytes())

manifest = {
    'notebook': 'colab/DADA2000Origin/phase_5_zscore.ipynb',
    'plan': '.project/plans/katvad-flow-zscore-option-a.md',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'core_sha256': digest.hexdigest()[:16],
    'torch': torch.__version__, 'transformers': transformers.__version__,
    'flow_v2': {'dir': str(FLOW_V2_ROOT / DATASET), 'V': V2, 'lambda_rec': LAMBDA_REC,
                'gates': {k: g['passed'] for k, g in ZM['gates'].items()},
                'between_item_share': T_V2['between_item_share']},
    'v1_reference': V1,
    'r1': {'kip_rec': K_V2, 'r2': R1},
    'r1b': R1B,
    'r2': {'rho_stage2': RHO, 'captured': R2_CAPTURED},
    'rows': ROWS,
    'verdict': VERDICT,
    'next': NEXT,
    'pairing': 'KIP-off reused from outputs/*_phase4; configs differ in kip.enabled + loss.lambda_rec only',
    'c24': 'untouched: every KIP-on arm is a fixed ~50 % channel shift on main',
}
(dest / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')
print('recorded ->', dest)
for p in sorted(dest.iterdir()):
    print(f'   {p.name}  ({p.stat().st_size / 1024:.1f} KiB)')
print(f'\nverdict: {VERDICT}\nnext:    {NEXT}')

---

## What to bring back

* `$KATVAD_OUTPUT_ROOT/REPORTS/DADA2000_orig_zscore/` — `run_manifest.json`, `zscore_manifest.json`, every
  `results_*.json` and `config_*.yaml`;
* `$KATVAD_OUTPUT_ROOT/DADA2000_orig_zscore/r2_grad/s*/grad_probe.{json,md}` and `p2_flow_target_v2/`.

Report **V_v2, lambda_rec, the gates, R-1, R-1b, rho per seed, the four Δ rows and the verdict**. They go into
the plan's Appendix A as measured, beside bars that were written before this notebook ran.